In [ ]:
from src.utils import AIDatasetLoader, filter_by_app_and_model, DecisionTreeInterpreter, LogisticRegressionInterpreter  
from src.memory import Chunk, DeclarativeMemory

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# # 🔧 Choose dataset here:
# app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# # 🧠 Auto-configured values:
# model_name = dataset_model_map[app_id]

# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
# dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=2)
# dt_exp.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
# lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant="sparse")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import importlib
import math
import os
import numpy as np
import pandas as pd
from typing import Iterable, Union, Optional
import random
from dataclasses import dataclass, replace as dc_replace

import src.memory as memory
importlib.reload(memory)
import src.dt_memory as dt_memory
importlib.reload(dt_memory)
import src.heuristic_lr_model as heuristic_lr_model
importlib.reload(heuristic_lr_model)
import src.lr_memory as lr_memory
importlib.reload(lr_memory)

from src.memory import DeclarativeMemory, CombinedMemory
from src.dt_memory import (
    add_node_to_memory, predict_with_dt_memory_prob, read_dt_prob,
    remember_with_dt_feedback, get_feature_nums
)
from src.heuristic_lr_model import (
    add_heuristic_lr_prob_to_memory, predict_with_heuristic_lr_prob_memory,
    remember_with_heuristic_lr_prob_feedback
)
from src.lr_memory import (
    add_lr_to_memory, predict_with_lr_memory_prob, read_lr_prob, remember_with_lr_feedback
)

In [ ]:
def _make_memory(retrieval_threshold, latency_factor):
    """
    Build a CombinedMemory(DeclarativeMemory) with typical defaults.
    Requires your environment to define DeclarativeMemory and CombinedMemory.
    """
    dm = DeclarativeMemory(
        retrieval_threshold=retrieval_threshold,
        latency_factor=latency_factor,
        latency_exponent=0.5,
        max_assoc_strength=2.0,
        mismatch_penalty=-1.0,
        activation_noise=0.1,
        decay=0.5,
    )
    return CombinedMemory(dm, wm_capacity=0)

def _call_with_known_kwargs(fn, *args, **kwargs):
    """Call fn but pass only kwargs it actually accepts (robust to signature drift)."""
    import inspect
    allowed = set(inspect.signature(fn).parameters.keys())
    clean = {k: v for k, v in kwargs.items() if k in allowed}
    return fn(*args, **clean)

def _maybe_get_feature_nums(explainer):
    try:
        return get_feature_nums(explainer)  # if your env provides it
    except Exception:
        # try common attributes
        if hasattr(explainer, "feature_nums"):
            return explainer.feature_nums
        if hasattr(explainer, "feature_names_"):
            return list(range(len(explainer.feature_names_)))
        return None


def _simulate_trials(
    T_READ_NUM, retrieval_threshold, latency_factor, W0_ANS, lapse,
    ai_dataset_loader, explainer, model_type: str, model_name: str,
    *, app_id: str, cogModel: str = "heuristic", active_indices = None, num_samples: int = 40,
    compute_sf: int = 2,   # <<< NEW
):
    import numpy as np

    mt = str(model_type).lower()
    cm = str(cogModel).lower()

    memory = _make_memory(retrieval_threshold, latency_factor)
    local_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name) if app_id is not None else ai_dataset_loader

    if mt == "dt":
        _call_with_known_kwargs(add_node_to_memory, explainer.tree_structure[0], memory,
                                dt_exp=explainer, feature_nums=_maybe_get_feature_nums(explainer))
    else:
        if cm == "calculation":
            _call_with_known_kwargs(add_lr_to_memory, explainer, memory)
        else:
            _call_with_known_kwargs(add_heuristic_lr_prob_to_memory, explainer, memory,
                                    initial_instance=None, initial_sigma=0.5)

    memory.tick(90)

    if active_indices is None:
        active_indices = [0, 1, 2, 3, 4, 5]

    # prob_list, rt_pred, rt_true = [], [], []
    

    all_instance_ids = list(range(400))
    selected_instance_ids = random.sample(all_instance_ids, k=40)

    w_xai_list = ["w/ XAI"] * 20 + ["w/o XAI"] * 20
    random.shuffle(w_xai_list)

    selections = []
    pred_times = []
    ai_preds = []
    explainer_preds = []

    for instance_id, with_xai in zip(selected_instance_ids, w_xai_list):
        # instance_id   = row['Instance Id']
        # with_xai_raw  = row.get('Tested w/ XAI', 'w/o XAI')
        # actual_resp   = row["Response"]
        # response_time = float(row["Time"])
        # with_xai = (with_xai_raw.strip().lower() in ("w/ xai", "with xai", "xai", "1", "true", "yes")) if isinstance(with_xai_raw, str) else bool(with_xai_raw)

        with_xai = (with_xai.strip().lower() in ("w/ xai", "with xai", "xai", "1", "true", "yes")) if isinstance(with_xai, str) else bool(with_xai)

        instances, _preds = _call_with_known_kwargs(local_loader.load_instances, [int(instance_id)], normalize=False)
        pred = _preds[0]
        ai_preds.append(pred)
        instance = instances[0]
        norm_instances, _ = _call_with_known_kwargs(local_loader.load_instances, [int(instance_id)], normalize=True)
        norm_instance = norm_instances[0]

        if mt == "dt":
            explainer_pred = int(explainer.apply_to_instance(instance)["class_index"])
        else:
            explainer_pred = 1 if float(explainer.apply_to_instance(instance)) > 0 else 0

        if with_xai:
            if mt == "dt":
                probs, pred_time, _ = _call_with_known_kwargs(
                    read_dt_prob, explainer, instance, T_READ_NUM=T_READ_NUM, W0_ANS=W0_ANS, lapse=lapse
                )
                _call_with_known_kwargs(remember_with_dt_feedback, instance, memory,
                                        feature_nums=_maybe_get_feature_nums(explainer))
            else:
                if cm == "calculation":
                    probs, pred_time, _ = _call_with_known_kwargs(
                        read_lr_prob, instance, explainer, T_READ_NUM=T_READ_NUM, W0_ANS=W0_ANS,
                        active_indices=active_indices, compute_sf=compute_sf, lapse=lapse   # <<< HERE
                    )
                    _call_with_known_kwargs(remember_with_lr_feedback, memory, explainer)
                else:
                    probs, pred_time, info = _call_with_known_kwargs(
                        predict_with_heuristic_lr_prob_memory, norm_instance, memory,
                        lr_exp=explainer, T_READ_NUM=T_READ_NUM, W0_ANS=W0_ANS,
                        active_indices=active_indices, num_samples=num_samples, lapse=lapse
                    )
                    _call_with_known_kwargs(
                        remember_with_heuristic_lr_prob_feedback, memory, explainer,
                        norm_instance, explainer_pred, int(np.argmax(probs)),
                        mean_z=info.get('mean_z', None), W0_ANS=W0_ANS,
                        active_indices=active_indices, learning_rate=0.2
                    )
        else:
            if mt == "dt":
                probs, pred_time, _ = _call_with_known_kwargs(
                    predict_with_dt_memory_prob, instance, memory,
                    dt_exp=explainer, T_READ_NUM=T_READ_NUM, W0_ANS=W0_ANS, lapse=lapse
                )
            else:
                if cm == "heuristic":
                    probs, pred_time, info = _call_with_known_kwargs(
                        predict_with_heuristic_lr_prob_memory, norm_instance, memory,
                        lr_exp=explainer, T_READ_NUM=T_READ_NUM, W0_ANS=W0_ANS,
                        active_indices=active_indices, num_samples=num_samples, lapse=lapse
                    )
                    _call_with_known_kwargs(
                        remember_with_heuristic_lr_prob_feedback, memory, explainer,
                        norm_instance, pred, int(np.argmax(probs)),
                        mean_z=(info.get('mean_z', None) if isinstance(info, dict) else None),
                        W0_ANS=W0_ANS, active_indices=active_indices, learning_rate=0.2
                    )
                else:
                    probs, pred_time, _ = _call_with_known_kwargs(
                        predict_with_lr_memory_prob, instance, memory,
                        lr_exp=explainer, T_READ_NUM=T_READ_NUM, W0_ANS=W0_ANS,
                        active_indices=active_indices, compute_sf=compute_sf, lapse=lapse  # <<< HERE
                    )

        # p = _select_prob(probs, actual_resp)
        # prob_list.append(float(p))
        # rt_pred.append(float(pred_time))
        # rt_true.append(float(response_time))

        # sample between 0 and 1 based on probs
        selection = random.choices([0, 1], weights=probs)[0]

        selections.append(selection)
        pred_times.append(float(pred_time))

        explainer_preds.append(explainer_pred)

    return app_id, selected_instance_ids, w_xai_list, ai_preds, explainer_preds, selections, pred_times


In [ ]:
params = {
    "W0_ANS": 0.1,
    "lapse": 0.05,
    "T_READ": 2.0, 
    "retrieval_threshold": 1.4,
    "latency_factor": 5.0,
    "compute_sf": 2,
}


# cfg = ObjConfig(
#     w_resp=1.0, w_time=0.1, t_df=3.0, repeats=1,
#     TREAD_bounds=(1.3, 1.5), RT_bounds=(-0.6, -0.5),
#     LAT_bounds=(0.4, 0.5), W0_bounds=(0.05, 0.15), lapse_bounds=(0.05, 0.2)
# )


In [ ]:
explainer_dict = {}
for complexity in ["low", "high"]:
    explainer_dict[("dt", complexity)] = DecisionTreeInterpreter(dt_df, metadata_df, "wine_quality", "mlp", depth=2 if complexity == "low" else 3)
    explainer_dict[("lr", complexity)] = LogisticRegressionInterpreter(lr_df, metadata_df, "wine_quality", "mlp", variant="sparse" if complexity == "low" else "dense")

num_participants = 25

p_id = 0

cogModels = {"dt": ["traverse"], "lr": ["heuristic", "calculation"]}

all_data = []

for model_type in ["dt", "lr"]:
    for complexity in ["low", "high"]:
        for cogModel in cogModels[model_type]:
            for i in range(1, num_participants + 1):
                print(p_id)
                p_id += 1
                app_id = "wine_quality"
                sim_result = _simulate_trials(
                    T_READ_NUM=params["T_READ"], retrieval_threshold=params["retrieval_threshold"],
                    latency_factor=params["latency_factor"], W0_ANS=params["W0_ANS"], lapse=params["lapse"],
                    ai_dataset_loader=ai_dataset_loader, explainer=explainer_dict[(model_type, complexity)],
                    model_type=model_type, model_name="mlp",
                    app_id=app_id, cogModel=cogModel, active_indices=None, num_samples=40,
                    compute_sf=params["compute_sf"],
                )
                app_id, instance_ids, w_xai_list, ai_preds, explainer_preds, selections, pred_times = sim_result
                for i, (inst_id, with_xai, sel, rt) in enumerate(zip(instance_ids, w_xai_list, selections, pred_times)):
                    all_data.append({
                        "Participant ID": p_id,
                        "AppId": app_id,
                        "Condition": model_type,
                        "Complexity": complexity,
                        "Instance Id": inst_id,
                        "Tested w/ XAI": with_xai,
                        "Response": sel,
                        "Time": rt,
                        "AI Prediction": ai_preds[i],
                        "Explainer Prediction": explainer_preds[i],
                        "Log(time)": math.log(rt + 1e-5),
                        "Strategy": cogModel,
                        "Response==AI": int(sel == ai_preds[i]),
                        "Response==Explainer": int(sel == explainer_preds[i]),
                    })

In [ ]:
#save all_data to csv
df = pd.DataFrame(all_data)
df.to_csv(f"forward_simulation_{app_id}_v_0.2.csv", index=False)